[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/22.%20The%20Internal%20Vehicle%20%28Terminal%20Truck%29%20Dispatching%20Problem/P22-Tier-2_executed.ipynb)

---



# 22. The Internal Vehicle (Terminal Truck) Dispatching Problem
## Tier 2: Auction-Based Heuristic Algorithm

### Goal and scope
This notebook implements a one-round, auction-based greedy heuristic for assigning container transport jobs to terminal trucks. It calculates a score for each feasible container–truck pair, sorts the scores from highest to lowest, and accepts a pair when neither its container nor its truck has already been assigned in that round.

The input is a set of containers, a set of trucks with current locations and availability times, and travel times between locations. The output is a set of one-to-one assignments, their scores, and related diagnostic summaries. The example is a small, static dispatch snapshot; execution time on this example does not establish real-time performance for larger instances.

### Model assumptions
- Each truck can receive at most one container in an auction round, and each container can be assigned to at most one truck in that round.
- Travel times are deterministic and are given for every ordered pair of distinct locations used in the example. The example data are symmetric.
- A container's `earliest_pickup` is the earliest time at which pickup may occur, and `latest_delivery` is the latest allowed delivery completion time.
- A truck may wait before beginning its trip so that pickup does not occur before the earliest pickup time.
- Container capacity, travel-time uncertainty, route congestion, loading and unloading durations, and new arrivals during a round are outside this example's scope.

### How the heuristic works
1. Calculate a bid score for every container–truck pair that can meet the container's delivery deadline.
2. Sort feasible bids in descending score order.
3. Scan the sorted list once, accepting a bid only if both its container and truck are still unassigned.
4. Report the resulting assignments and inspect their scores, feasibility, and sensitivity to selected inputs.

A higher score expresses the heuristic's preference; it is not a monetary payment. The score combines priority and time urgency, pickup efficiency, and a completion-time penalty. Greedy acceptance makes the procedure easy to inspect, but the score and selection rule alone do not establish a general approximation guarantee or prove that the result is optimal.

## 1. Bid score and assignment rule

A bid is calculated only when the truck can reach the container's origin, the container's destination is reachable from its origin, and delivery can be completed by the deadline. Let $c$ denote a container, $v$ a truck, and $t$ the current dispatch time. Define $p_{vc}$ as the travel time from truck $v$'s current location to container $c$'s origin, and $q_c$ as the travel time from that origin to the destination. The truck's departure time accounts for its availability, the current dispatch time, and the earliest pickup time:

$$
s_{vc}(t) = \max\left\{a_v,\ t,\ e_c - p_{vc}\right\},
\qquad
F_{vc}(t) = s_{vc}(t) + p_{vc} + q_c.
$$

Here, $a_v$ is the truck's available time, $e_c$ is the earliest pickup time, and $F_{vc}(t)$ is the delivery completion time. Subtracting the pickup travel time from $e_c$ allows the truck to depart early enough to arrive at the origin no earlier than the permitted pickup time. The pair is feasible when both required travel times are available and $F_{vc}(t) \leq d_c$, where $d_c$ is the latest delivery time.

For a feasible pair, the notebook uses the following score:

$$
b_{vc}(t)
=
1000\left(\frac{r_c}{\max\{d_c-t,\ 1\}}\right)
\left(\frac{1}{p_{vc}+1}\right)
- F_{vc}(t).
$$

In this expression, $r_c$ is the container priority. The first factor rewards high-priority containers with little time remaining; the second favors trucks with shorter pickup trips; the final term penalizes later completion times. The constant 1000 is a score-weighting parameter in the given heuristic. An infeasible pair receives a score of $-\infty$ and is excluded from the auction.

Let $C$ be the set of containers and $V$ the set of trucks. The auction forms the feasible bid list

$$
\mathcal{B}(t)
=
\left\{(b_{vc}(t),c,v): c\in C,\ v\in V,\ F_{vc}(t)\leq d_c\right\},
$$

with unreachable pairs excluded as well. After sorting this list in descending score order, the greedy rule accepts a pair if neither member has already been accepted. If $x_{vc}=1$ means that container $c$ is assigned to truck $v$ in this round, the resulting assignments satisfy

$$
\sum_{v\in V}x_{vc}\leq 1\quad(c\in C),
\qquad
\sum_{c\in C}x_{vc}\leq 1\quad(v\in V).
$$

These inequalities describe the one-to-one assignment rule; the implementation constructs assignments directly by tracking which containers and trucks have already been selected. The total system utility reported here is the sum of the selected bid scores, not an independently optimized objective.

If $|C|$ is the number of containers, $|V|$ the number of trucks, and $m\leq |C||V|$ the number of feasible pairs, bid generation takes $O(|C||V|)$ time and sorting takes $O(m\log m)$. The greedy scan takes $O(m)$ time. Thus the total time is $O(|C||V|+m\log m)$, with $O(m)$ space for the bid list. The procedure is a heuristic: no approximation ratio is claimed for this score and greedy rule.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Setting up auction-based terminal truck dispatching...")

In [ ]:
@dataclass
class Container:
    """A container requiring transport service."""

    id: str
    origin: str
    destination: str
    priority: int
    earliest_pickup: int
    latest_delivery: int


@dataclass
class Truck:
    """A terminal truck with a current location and availability time."""

    id: str
    current_location: str
    available_time: int
    capacity: int = 1


class TerminalDispatcher:
    """Generate pairwise bids and assign jobs with a greedy auction round."""

    def __init__(self, travel_times: Dict[Tuple[str, str], int]):
        self.travel_times = travel_times
        self.assignment_history = []

    def calculate_bid_value(
        self, container: Container, truck: Truck, current_time: int
    ) -> float:
        """Return the heuristic score, or negative infinity if infeasible."""
        pickup_time = self.travel_times.get(
            (truck.current_location, container.origin), float("inf")
        )
        transport_time = self.travel_times.get(
            (container.origin, container.destination), float("inf")
        )

        if pickup_time == float("inf") or transport_time == float("inf"):
            return -float("inf")

        # Depart late enough to respect the pickup window while honoring availability.
        start_time = max(
            truck.available_time,
            current_time,
            container.earliest_pickup - pickup_time,
        )
        completion_time = start_time + pickup_time + transport_time

        if completion_time > container.latest_delivery:
            return -float("inf")

        time_remaining = container.latest_delivery - current_time
        urgency = container.priority / max(time_remaining, 1)
        efficiency = 1.0 / (pickup_time + 1)
        return urgency * efficiency * 1000 - completion_time

    def auction_round(
        self,
        containers: List[Container],
        trucks: List[Truck],
        current_time: int,
    ) -> Dict[str, str]:
        """Run one auction round and return container-to-truck assignments."""
        bids = []
        for container in containers:
            for truck in trucks:
                bid_value = self.calculate_bid_value(
                    container, truck, current_time
                )
                if bid_value > -float("inf"):
                    bids.append(
                        (bid_value, container.id, truck.id, container, truck)
                    )

        # A stable descending sort makes equal-score handling reproducible
        # for a fixed input order.
        bids.sort(key=lambda bid: bid[0], reverse=True)

        assignments = {}
        assigned_trucks = set()
        assigned_containers = set()

        for bid_value, container_id, truck_id, container, truck in bids:
            if truck_id in assigned_trucks or container_id in assigned_containers:
                continue

            assignments[container_id] = truck_id
            assigned_trucks.add(truck_id)
            assigned_containers.add(container_id)
            self.assignment_history.append(
                {
                    "container_id": container_id,
                    "truck_id": truck_id,
                    "bid_value": bid_value,
                    "priority": container.priority,
                    "origin": container.origin,
                    "destination": container.destination,
                    "completion_time": self._calculate_completion_time(
                        container, truck, current_time
                    ),
                }
            )

        return assignments

    def _calculate_completion_time(
        self, container: Container, truck: Truck, current_time: int
    ) -> int:
        """Calculate completion time using the same timing rule as the bid."""
        pickup_time = self.travel_times[(truck.current_location, container.origin)]
        transport_time = self.travel_times[(container.origin, container.destination)]
        start_time = max(
            truck.available_time,
            current_time,
            container.earliest_pickup - pickup_time,
        )
        return start_time + pickup_time + transport_time

    def update_truck_status(
        self, truck: Truck, container: Container, current_time: int
    ) -> None:
        """Move an assigned truck to the job destination and update availability."""
        completion_time = self._calculate_completion_time(
            container, truck, current_time
        )
        truck.available_time = completion_time
        truck.current_location = container.destination

    def calculate_system_utility(
        self,
        assignments: Dict[str, str],
        containers: List[Container],
        trucks: List[Truck],
        current_time: int,
    ) -> float:
        """Sum the bid scores for assignments at the supplied dispatch snapshot."""
        containers_by_id = {container.id: container for container in containers}
        trucks_by_id = {truck.id: truck for truck in trucks}
        total_utility = 0.0

        for container_id, truck_id in assignments.items():
            bid_value = self.calculate_bid_value(
                containers_by_id[container_id],
                trucks_by_id[truck_id],
                current_time,
            )
            if bid_value == -float("inf"):
                raise ValueError("An assignment is infeasible at this snapshot.")
            total_utility += bid_value

        return total_utility


print("Data structures and dispatcher class are ready.")

## 2. Define the dispatch instance

The example contains four container jobs and three trucks. Travel time is measured in integer time units. Each container has a priority, an earliest pickup time, and a latest delivery time. These data determine both whether a pair is feasible and the score it receives.

In [ ]:
travel_times = {
    ("berth", "storage"): 2,
    ("storage", "berth"): 2,
    ("storage", "rail"): 3,
    ("rail", "storage"): 3,
    ("storage", "gate"): 2,
    ("gate", "storage"): 2,
    ("berth", "rail"): 4,
    ("rail", "berth"): 4,
    ("berth", "gate"): 3,
    ("gate", "berth"): 3,
    ("rail", "gate"): 3,
    ("gate", "rail"): 3,
}

containers = [
    Container("C1", "berth", "storage", 10, 0, 8),
    Container("C2", "storage", "rail", 15, 1, 7),
    Container("C3", "storage", "gate", 8, 0, 10),
    Container("C4", "berth", "gate", 12, 2, 9),
]

trucks = [
    Truck("T1", "berth", 0),
    Truck("T2", "storage", 0),
    Truck("T3", "gate", 0),
]

locations = sorted(
    {container.origin for container in containers}
    | {container.destination for container in containers}
)

print("Problem setup:")
print(f"- Containers: {len(containers)}")
print(f"- Trucks: {len(trucks)}")
print(f"- Locations: {locations}")
print(f"- Directed travel-time entries: {len(travel_times)}")
print("\nContainer details:")
for container in containers:
    print(
        f"  {container.id}: {container.origin} to {container.destination} "
        f"(priority: {container.priority}, "
        f"pickup no earlier than {container.earliest_pickup}, "
        f"delivery deadline: {container.latest_delivery})"
    )

## 3. Run one auction round

All bids in a round use the same initial truck locations and availability times. The greedy scan then selects non-conflicting pairs from the sorted bids. Truck status updates are demonstrated on separate copies after the round, so the diagnostic tables and plots continue to describe the same dispatch snapshot.

In [ ]:
dispatcher = TerminalDispatcher(travel_times)
current_time = 0

print("Executing one auction-based dispatch round...")
print("=" * 52)

start_time = time.perf_counter()
assignments = dispatcher.auction_round(containers, trucks, current_time)
execution_time = time.perf_counter() - start_time
total_utility = dispatcher.calculate_system_utility(
    assignments, containers, trucks, current_time
)

containers_by_id = {container.id: container for container in containers}
trucks_by_id = {truck.id: truck for truck in trucks}

assert len(assignments) == len(set(assignments))
assert len(assignments.values()) == len(set(assignments.values()))

print(f"Auction computation time: {execution_time:.6f} seconds")
print(f"Assignments made: {len(assignments)} of {len(containers)} containers")

if assignments:
    print("\nSelected assignments:")
    print(
        f"{'Container':<12} {'Truck':<8} {'Bid score':<12} "
        f"{'Route':<22} {'Completion':<12}"
    )
    print("-" * 72)

    for container_id, truck_id in assignments.items():
        container = containers_by_id[container_id]
        truck = trucks_by_id[truck_id]
        bid_value = dispatcher.calculate_bid_value(
            container, truck, current_time
        )
        completion_time = dispatcher._calculate_completion_time(
            container, truck, current_time
        )
        route = f"{container.origin} to {container.destination}"
        print(
            f"{container_id:<12} {truck_id:<8} {bid_value:<12.2f} "
            f"{route:<22} {completion_time:<12}"
        )

    print(f"\nTotal system utility (sum of selected bid scores): {total_utility:.2f}")
else:
    print("No feasible assignments were found.")

# Independent copies show how accepted jobs update truck state for a later round.
updated_trucks = {
    truck.id: Truck(
        truck.id,
        truck.current_location,
        truck.available_time,
        truck.capacity,
    )
    for truck in trucks
}
for container_id, truck_id in assignments.items():
    dispatcher.update_truck_status(
        updated_trucks[truck_id], containers_by_id[container_id], current_time
    )

print("\nTruck status after the assigned jobs:")
for truck in trucks:
    updated_truck = updated_trucks[truck.id]
    print(
        f"  {truck.id}: location {updated_truck.current_location}, "
        f"available at time {updated_truck.available_time}"
    )

## 4. Inspect feasible bids

The bid table exposes the pairwise calculations that feed the greedy selection. A feasible bid is not necessarily selected: a higher-ranked bid may already have used its truck or container. Comparing the selected pairs with the full table makes this conflict-resolution rule visible.

In [ ]:
bid_analysis = []
for container in containers:
    for truck in trucks:
        bid_value = dispatcher.calculate_bid_value(
            container, truck, current_time
        )
        if bid_value == -float("inf"):
            continue

        pickup_time = travel_times[(truck.current_location, container.origin)]
        transport_time = travel_times[(container.origin, container.destination)]
        completion_time = dispatcher._calculate_completion_time(
            container, truck, current_time
        )
        bid_analysis.append(
            {
                "Container": container.id,
                "Truck": truck.id,
                "Truck location": truck.current_location,
                "Route": f"{container.origin} to {container.destination}",
                "Priority": container.priority,
                "Pickup travel time": pickup_time,
                "Transport time": transport_time,
                "Completion time": completion_time,
                "Bid score": bid_value,
            }
        )

bid_df = pd.DataFrame(bid_analysis)
if not bid_df.empty:
    bid_df_sorted = bid_df.sort_values(
        "Bid score", ascending=False, kind="stable"
    )
    print("Feasible bids, sorted by score:")
    print(bid_df_sorted.round(2).to_string(index=False))

    print("\nSelected bids:")
    selected_rows = bid_df_sorted[
        bid_df_sorted.apply(
            lambda row: assignments.get(row["Container"]) == row["Truck"],
            axis=1,
        )
    ]
    if selected_rows.empty:
        print("No bids were selected.")
    else:
        for _, row in selected_rows.iterrows():
            print(
                f"  {row['Container']} assigned to {row['Truck']}: "
                f"score {row['Bid score']:.2f}"
            )
else:
    print("No feasible bids were generated.")

## 5. Interpret computational effort and solution quality

The complexity estimate follows the work performed by the code: at most one bid is calculated for each container–truck pair, feasible bids are sorted, and the sorted list is scanned once. The measured time is specific to this run and computing environment; it is not evidence by itself that the method meets an operational deadline. The assignment rate counts served containers, while system utility is the sum of their heuristic scores.

In [ ]:
number_of_pairs = len(containers) * len(trucks)
number_of_feasible_bids = len(bid_df)

print("Auction performance summary")
print("=" * 52)
print("\n1. Work performed:")
print(f"   - Containers: {len(containers)}")
print(f"   - Trucks: {len(trucks)}")
print(f"   - Candidate container-truck pairs: {number_of_pairs}")
print(f"   - Feasible bids sorted: {number_of_feasible_bids}")
print("   - Bid generation: O(|C| x |V|)")
print("   - Sorting and greedy scan: O(m log m) and O(m)")
print("   - Total: O(|C||V| + m log m), where m is the feasible-bid count")

print("\n2. Quality and timing for this instance:")
print(f"   - Auction computation time: {execution_time:.6f} seconds")
print(f"   - Containers assigned: {len(assignments)} of {len(containers)}")
if containers:
    assignment_rate = len(assignments) / len(containers)
    print(f"   - Assignment rate: {assignment_rate:.1%}")
if assignments:
    print(f"   - Sum of selected bid scores: {total_utility:.2f}")
    print(
        "   - Mean selected bid score: "
        f"{total_utility / len(assignments):.2f}"
    )

print("\nThe greedy rule is a heuristic; no general approximation ratio is claimed.")

## 6. Visualize the auction results

The four panels show the distribution of feasible bid scores, the relationship between priority and score, a truck–container score matrix, and the time interval from truck dispatch through delivery. Gray cells in the matrix represent infeasible pairs; a star marks a selected assignment. The timeline uses the initial truck snapshot and the same completion-time calculation as the bid.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    2, 2, figsize=(16, 12)
)

if not bid_df.empty:
    ax1.hist(
        bid_df["Bid score"],
        bins=min(10, max(1, len(bid_df))),
        alpha=0.7,
        color="skyblue",
        edgecolor="black",
    )
    ax1.set_title("Distribution of feasible bid scores", fontweight="bold")
    ax1.set_xlabel("Bid score")
    ax1.set_ylabel("Number of pairs")
    ax1.grid(True, alpha=0.3)

    selected_scores = [
        dispatcher.calculate_bid_value(
            containers_by_id[container_id],
            trucks_by_id[truck_id],
            current_time,
        )
        for container_id, truck_id in assignments.items()
    ]
    if selected_scores:
        ax1.hist(
            selected_scores,
            bins=min(5, max(1, len(selected_scores))),
            alpha=0.75,
            color="tomato",
            edgecolor="black",
            label="Selected bids",
        )
        ax1.legend()

    container_colors = plt.cm.tab10(
        np.linspace(0, 1, max(len(containers), 1))
    )
    for index, container in enumerate(containers):
        container_bids = bid_df[bid_df["Container"] == container.id]
        if container_bids.empty:
            continue
        ax2.scatter(
            container_bids["Priority"],
            container_bids["Bid score"],
            s=90,
            color=container_colors[index],
            alpha=0.8,
            label=container.id,
        )
        if container.id in assignments:
            selected_bid = container_bids[
                container_bids["Truck"] == assignments[container.id]
            ]
            ax2.scatter(
                selected_bid["Priority"],
                selected_bid["Bid score"],
                s=220,
                color=[container_colors[index]],
                edgecolor="black",
                linewidth=1.5,
                marker="*",
            )
    ax2.set_title("Priority and bid score", fontweight="bold")
    ax2.set_xlabel("Container priority")
    ax2.set_ylabel("Bid score")
    ax2.grid(True, alpha=0.3)
    ax2.legend(title="Container")
else:
    ax1.set_title("No feasible bids")
    ax2.set_title("No feasible bids")

assignment_matrix = np.full(
    (len(trucks), len(containers)), np.nan, dtype=float
)
for truck_index, truck in enumerate(trucks):
    for container_index, container in enumerate(containers):
        bid_value = dispatcher.calculate_bid_value(
            container, truck, current_time
        )
        if bid_value != -float("inf"):
            assignment_matrix[truck_index, container_index] = bid_value / 1000

masked_matrix = np.ma.masked_invalid(assignment_matrix)
cmap = plt.cm.RdYlBu_r.copy()
cmap.set_bad("lightgray")
image = ax3.imshow(masked_matrix, cmap=cmap, aspect="auto")
ax3.set_title("Truck-container bid score matrix", fontweight="bold")
ax3.set_xlabel("Containers")
ax3.set_ylabel("Trucks")
ax3.set_xticks(range(len(containers)))
ax3.set_yticks(range(len(trucks)))
ax3.set_xticklabels([container.id for container in containers])
ax3.set_yticklabels([truck.id for truck in trucks])
plt.colorbar(image, ax=ax3, label="Feasible bid score / 1000")
for truck_index, truck in enumerate(trucks):
    for container_index, container in enumerate(containers):
        if assignments.get(container.id) == truck.id:
            ax3.scatter(
                container_index,
                truck_index,
                marker="*",
                s=220,
                color="black",
                label="Selected assignment",
            )
handles, labels = ax3.get_legend_handles_labels()
if handles:
    ax3.legend(handles[:1], labels[:1], loc="best")

ax4.set_title("Dispatch-to-delivery timeline", fontweight="bold")
ax4.set_xlabel("Time units")
ax4.set_ylabel("Assigned container")
ax4.grid(True, axis="x", alpha=0.3)
for row_index, (container_id, truck_id) in enumerate(assignments.items()):
    container = containers_by_id[container_id]
    truck = trucks_by_id[truck_id]
    pickup_time = travel_times[(truck.current_location, container.origin)]
    departure_time = max(
        truck.available_time,
        current_time,
        container.earliest_pickup - pickup_time,
    )
    completion_time = dispatcher._calculate_completion_time(
        container, truck, current_time
    )
    ax4.barh(
        row_index,
        completion_time - departure_time,
        left=departure_time,
        height=0.6,
        alpha=0.8,
        label=f"{container_id} assigned to {truck_id}",
    )
    ax4.text(
        departure_time + (completion_time - departure_time) / 2,
        row_index,
        f"{departure_time} to {completion_time}",
        ha="center",
        va="center",
        fontsize=9,
    )

ax4.set_yticks(range(len(assignments)))
ax4.set_yticklabels(list(assignments.keys()))
if assignments:
    ax4.legend(title="Assignment", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

## 7. What-if sensitivity analysis

Sensitivity analysis reruns the same auction procedure on modified copies of the inputs. The first set of scenarios changes container priorities; the second scales the travel-time entries. Each scenario starts with fresh trucks, so one run does not carry truck updates into another. Travel times remain integers: multiplying an entry by a scenario factor and converting it to `int` truncates any fractional part.

In [ ]:
print("Priority sensitivity scenarios")
priority_scenarios = [
    {"name": "Original", "priorities": [10, 15, 8, 12]},
    {"name": "Equal priority", "priorities": [10, 10, 10, 10]},
    {"name": "High priority for C1", "priorities": [25, 15, 8, 12]},
    {"name": "Reversed priority", "priorities": [8, 12, 15, 25]},
]

scenario_results = []
for scenario in priority_scenarios:
    test_containers = [
        Container(
            container.id,
            container.origin,
            container.destination,
            scenario["priorities"][index],
            container.earliest_pickup,
            container.latest_delivery,
        )
        for index, container in enumerate(containers)
    ]
    test_trucks = [
        Truck(truck.id, truck.current_location, truck.available_time, truck.capacity)
        for truck in trucks
    ]
    test_dispatcher = TerminalDispatcher(travel_times)
    test_assignments = test_dispatcher.auction_round(
        test_containers, test_trucks, current_time
    )
    test_utility = test_dispatcher.calculate_system_utility(
        test_assignments, test_containers, test_trucks, current_time
    )
    scenario_results.append(
        {
            "Scenario": scenario["name"],
            "Assignments": len(test_assignments),
            "Utility": test_utility,
            "Assignment rate": len(test_assignments) / len(test_containers),
        }
    )

scenario_df = pd.DataFrame(scenario_results)
print(scenario_df.round(2).to_string(index=False))

print("\nTravel-time sensitivity scenarios")
travel_time_scenarios = [
    {"name": "Original", "multiplier": 1.0},
    {"name": "Congested", "multiplier": 1.5},
    {"name": "Fast operations", "multiplier": 0.7},
]

for scenario in travel_time_scenarios:
    modified_travel_times = {
        route: int(travel_time * scenario["multiplier"])
        for route, travel_time in travel_times.items()
    }
    modified_dispatcher = TerminalDispatcher(modified_travel_times)
    fresh_trucks = [
        Truck(truck.id, truck.current_location, truck.available_time, truck.capacity)
        for truck in trucks
    ]
    modified_assignments = modified_dispatcher.auction_round(
        containers, fresh_trucks, current_time
    )
    modified_utility = modified_dispatcher.calculate_system_utility(
        modified_assignments, containers, fresh_trucks, current_time
    )
    print(
        f"{scenario['name']} (x{scenario['multiplier']}): "
        f"{len(modified_assignments)} assignments, "
        f"utility = {modified_utility:.2f}"
    )

## 8. Strengths, limitations, and appropriate use

### Strengths
- The score and the descending-bid selection rule are explicit, making individual decisions inspectable.
- For $|C|$ containers, $|V|$ trucks, and $m$ feasible pairs, the sorting-based implementation runs in $O(|C||V|+m\log m)$ time and stores $O(m)$ bids.
- A single scan constructs assignments that respect the one-container-per-truck and one-truck-per-container constraints for the round.
- The same procedure can be rerun on updated inputs, as illustrated by the sensitivity scenarios.

### Limitations
- A greedy choice can prevent a later combination of assignments with a larger total bid score. The procedure does not guarantee a globally optimal assignment.
- The heuristic's behavior depends on the chosen score, including its weights and time-urgency definition.
- This implementation schedules one job per truck in a round. It does not sequence multiple jobs for a truck within that round.
- The example does not establish a theoretical approximation ratio, a real-time service guarantee, or robustness to unmodeled operational conditions.

This approach is useful as a transparent dispatch heuristic when a fast, easily inspected assignment rule is desired. Its assignments should be interpreted with the stated scoring rule and scope in mind, rather than as proof of optimality or a general performance bound.

## Summary

The notebook builds a one-round auction-based dispatch heuristic from pairwise bid calculation through greedy conflict resolution. A bid is available only when the truck can reach the job and complete delivery within its deadline; the timing calculation also respects the earliest pickup time. Feasible bids are sorted in descending score order, and a bid is selected only when its truck and container are both unassigned.

The bid table, performance summary, visualization, and sensitivity scenarios provide several ways to inspect the behavior of the same procedure. The reported utility is the sum of selected heuristic scores. The algorithm's polynomial running time for this implementation does not imply optimality, a general approximation guarantee, or suitability for every real-time operation.